# Q3-02：考虑预测误差概率的 06/12 时滚动优化

本 notebook 不再将 06:00、12:00 得到的未来预报视作真实值。每次更新时：

1. 负荷使用 Q2 冻结模型的点预测，光伏使用附件 3 在当前时刻发布的预报；
2. 从此前最多 28 天、同一发布时间的**负荷和光伏联合预测残差**构造经验情景，保留单日内误差的时间相关性；
3. 按时间衰减权重赋予情景发生概率，并以 `期望紧急购电费用 + λ × CVaR(90%)` 进行风险敏感优化；
4. 只修改接下来的 6 小时计划，剩余时段仍锁定零点计划。

对比三组策略：不更新、将预测视作准确的理论上限、概率风险更新。最后一个 cell 给出评价期总费用，便于和 baseline 对照。


**费用口径修正**：零点目标为“普通计划购电费 + 期望紧急购电费”；日内原计划费为常数，只优化调整费、紧急购电费及该方案启用的风险项。评价期统一为 2025-02-01 至 2025-12-31。使用 Q2 预测的场景调度不等同于 Q2 的 0.85 分位数固定调度策略。


In [1]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import Bounds, LinearConstraint, linprog, milp
from scipy.sparse import coo_matrix, csr_matrix, hstack, vstack

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)

ROOT = Path.cwd()
while not (ROOT / 'C题.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

ACTUAL_PATH = ROOT / 'q2' / 'input' / 'q2_actual_load_pv.csv'
PRICE_PATH = ROOT / 'q2' / 'input' / 'q2_fixed_prices.csv'
FORECAST_PATH = ROOT / 'Data_preprocessed' / 'processed' / 'pv_forecast_10min.csv'
OUTPUT_DIR = ROOT / '.tmp' / 'q3_02_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    'interval_hours': 1 / 6,
    'charge_efficiency': 0.9,
    'discharge_efficiency': 0.9,
    'storage_min_kwh': 1200.0,
    'storage_max_kwh': 10800.0,
    'initial_storage_kwh': 6000.0,
    'planned_terminal_kwh': 6000.0,
    'max_charge_power_kw': 5000.0,
    'max_discharge_power_kw': 5000.0,
    'emergency_multiplier': 5.0,
    'adjustment_up_multiplier': 1.5,
    'adjustment_down_multiplier': 0.5,
    'history_days': 28,
    'probability_half_life_days': 14,
    'cvar_alpha': 0.90,
    'risk_lambda': 0.35,
    'tolerance': 1e-6,
}

STRATEGIES = {
    '对照：00时Q2计划不更新': 'none',
    '理论上限：06时+12时完美更新': 'perfect',
    '方案：06时+12时概率风险更新': 'probabilistic',
}

print('输出目录：', OUTPUT_DIR)
print('概率参数：', {k: CFG[k] for k in ['history_days', 'probability_half_life_days', 'cvar_alpha', 'risk_lambda']})


输出目录： /Users/jinyu/workspace/MCM/CUMCM_2026/.tmp/q3_02_outputs
概率参数： {'history_days': 28, 'probability_half_life_days': 14, 'cvar_alpha': 0.9, 'risk_lambda': 0.35}


## 1. 数据、附件预报与 Q2 初始预测

附件 3 只含光伏预报，因此日内更新的负荷中心预测来自 Q2；光伏中心预测来自附件 3 的 06:00 / 12:00 发布版本。

In [2]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


actual = pd.read_csv(ACTUAL_PATH, parse_dates=['date']).sort_values(['date', 'slot'])
prices_df = pd.read_csv(PRICE_PATH).sort_values('slot')
forecasts_df = pd.read_csv(FORECAST_PATH, parse_dates=['issue_time', 'interval_start', 'interval_end', 'anchor_available_at'])

require(len(actual) == 365 * 144, '实际数据应为365天×144个时段')
require(actual.groupby('date').size().eq(144).all(), '实际数据存在日期缺行')
require(not actual.duplicated(['date', 'slot']).any(), '实际数据存在重复日期-时段')
require(len(prices_df) == 144 and len(forecasts_df) > 0, '电价或附件3预报不完整')

dates = pd.DatetimeIndex(sorted(actual['date'].unique()))
load_kw = actual['load_kw'].to_numpy(float).reshape(len(dates), 144)
pv_kw = actual['pv_kw'].to_numpy(float).reshape(len(dates), 144)
prices = prices_df['price_yuan_per_kwh'].to_numpy(float)
require(np.isfinite(load_kw).all() and np.isfinite(pv_kw).all(), '实际功率含非有限值')

issue_hours = (0, 6, 12, 18)
forecasts = np.full((len(dates), len(issue_hours), 144), np.nan)
day_to_index = {pd.Timestamp(day): i for i, day in enumerate(dates)}
for (issue_time, issue_hour), block in forecasts_df.assign(issue_hour=forecasts_df['issue_time'].dt.hour).groupby(['issue_time', 'issue_hour']):
    day = pd.Timestamp(issue_time).normalize()
    if day not in day_to_index or int(issue_hour) not in issue_hours:
        continue
    k, j = day_to_index[day], issue_hours.index(int(issue_hour))
    start = int(issue_hour) * 6
    target = pd.date_range(day + pd.Timedelta(hours=int(issue_hour)), day + pd.Timedelta(days=1), freq='10min', inclusive='left')
    values = block.set_index('interval_start')['pv_forecast_kw_interpolated'].reindex(target)
    require(values.notna().all(), f'{issue_time} 预报无法覆盖当天剩余时段')
    forecasts[k, j, start:] = values.to_numpy(float)

for j, hour in enumerate(issue_hours):
    require(np.isfinite(forecasts[:, j, hour * 6:]).all(), f'{hour:02d}:00 光伏预报不完整')

import sys
sys.path.insert(0, str(ROOT / 'q2'))
from q2_final import FROZEN_FORECAST, forecast_scenarios

q2_pred_load_kw, q2_pred_pv_kw, q2_scenario_net_kw = forecast_scenarios(dates, load_kw, pv_kw)
q2_first_valid_day, q2_first_scenario_day = 14, 15
require(np.isfinite(q2_pred_load_kw[q2_first_valid_day:]).all(), 'Q2负荷预测无效')
require(all(k in q2_scenario_net_kw for k in range(q2_first_scenario_day, len(dates))), 'Q2场景不完整')
print('Q2 冻结参数：', FROZEN_FORECAST)
print('实际/附件预报数组：', load_kw.shape, forecasts.shape)


Q2 冻结参数： {'window_days': 60, 'k_load': 3, 'k_pv': 2, 'alpha_load': 0.3, 'alpha_pv': 10.0, 'scenario_days': 28}
实际/附件预报数组： (365, 144) (365, 4, 144)


## 2. 概率情景构造

每个历史日形成一条完整的残差轨迹，而非逐时独立抽样，因此保留连续阴天、持续偏高负荷等时序关联。距当前越近的历史日权重越大，权重半衰期为 14 天。

In [3]:
def q2_initial_scenarios(day_index):
    if day_index in q2_scenario_net_kw:
        x = q2_scenario_net_kw[day_index] * CFG['interval_hours']
        return x, np.full(len(x), 1 / len(x))
    # 仅用于1月预热；评价期不会使用真实值作为预测。
    x = ((load_kw[day_index] - pv_kw[day_index]) * CFG['interval_hours'])[None, :]
    return x, np.ones(1)


def update_scenarios(day_index, hour, perfect=False):
    """返回从更新时间到24:00的净负荷情景(kWh)及其概率。"""
    start = hour * 6
    if perfect:
        # 只把接下来6小时替换为真实净负荷；之后仍是零点Q2残差场景。
        scenarios, weights = q2_initial_scenarios(day_index)
        scenarios = scenarios[:, start:].copy()
        m = min(36, scenarios.shape[1])
        scenarios[:, :m] = ((load_kw[day_index, start:start+m] - pv_kw[day_index, start:start+m]) * CFG['interval_hours'])[None, :]
        return scenarios, weights

    if day_index < q2_first_valid_day:
        # Q2 点预测尚在预热，不能拿到概率模型中估计残差；沿用零点场景推进状态。
        scenarios, weights = q2_initial_scenarios(day_index)
        return scenarios[:, start:], weights

    issue_index = issue_hours.index(hour)
    first = max(q2_first_valid_day, day_index - CFG['history_days'])
    history = np.arange(first, day_index)
    if len(history) == 0:
        # 预热期的安全退化：以当前中心预测为唯一情景。
        centre = q2_pred_load_kw[day_index, start:] - forecasts[day_index, issue_index, start:]
        return (centre * CFG['interval_hours'])[None, :], np.ones(1)

    # 当前中心预测：Q2负荷 + 当前发布版本的附件3光伏预报。
    load_centre = q2_pred_load_kw[day_index, start:]
    pv_centre = forecasts[day_index, issue_index, start:]
    # 同一发布时间的联合残差：负荷误差与光伏误差在同一历史日配对。
    load_error = load_kw[history, start:] - q2_pred_load_kw[history, start:]
    pv_error = pv_kw[history, start:] - forecasts[history, issue_index, start:]
    scenarios = (load_centre[None, :] + load_error - np.maximum(pv_centre[None, :] + pv_error, 0.0)) * CFG['interval_hours']

    ages = day_index - history
    raw_weight = np.exp(-np.log(2) * ages / CFG['probability_half_life_days'])
    weights = raw_weight / raw_weight.sum()
    return scenarios, weights


demo_day = 60
for hour in (6, 12):
    demo_scenarios, demo_weights = update_scenarios(demo_day, hour)
    print(f'{hour:02d}:00：{len(demo_weights)} 个情景，最大单情景概率 {demo_weights.max():.2%}，未来6h净负荷 P10/P50/P90(kWh)=', np.quantile(demo_scenarios[:, :36].sum(axis=1), [.1, .5, .9]))


06:00：28 个情景，最大单情景概率 6.44%，未来6h净负荷 P10/P50/P90(kWh)= [11147.87821096 12281.16747835 13095.27869268]
12:00：28 个情景，最大单情景概率 6.44%，未来6h净负荷 P10/P50/P90(kWh)= [14142.2348494  14886.99394361 15853.28560872]


## 3. 规划模型：期望紧急购电费用 + CVaR 风险项

CVaR(90%) 表示最坏 10% 情景中的平均紧急购电费用。`risk_lambda` 越大，计划越保守。

In [4]:
def _storage_parts(net_scenarios, p, initial_kwh, fixed_grid=None, lock_from=None):
    a, p = np.asarray(net_scenarios, float), np.asarray(p, float)
    s, n = a.shape
    ec, ed = CFG['charge_efficiency'], CFG['discharge_efficiency']
    mc, md = CFG['max_charge_power_kw'] / 6, CFG['max_discharge_power_kw'] / 6
    # x=[g,c,d,E,h,u,v]，h为各情景、各时段的紧急购电量。
    h0, u0, v0 = 4*n + 1, 4*n + 1 + s*n, 4*n + 1 + s*n + n
    m = v0 + n
    obj = np.zeros(m)
    # 零点计划须计入普通购电费；日内原计划费已固定，只优化调整费与紧急费。
    if fixed_grid is None:
        obj[:n] = p
    lower, upper = np.zeros(m), np.full(m, np.inf)
    upper[n:2*n], upper[2*n:3*n] = mc, md
    lower[3*n:4*n+1], upper[3*n:4*n+1] = CFG['storage_min_kwh'], CFG['storage_max_kwh']
    lower[3*n] = upper[3*n] = initial_kwh
    lower[4*n] = upper[4*n] = CFG['planned_terminal_kwh']
    obj[h0:u0] = np.tile(CFG['emergency_multiplier'] * p / s, s)

    t = np.arange(n)
    storage_eq = coo_matrix((np.r_[-ec*np.ones(n), np.ones(n)/ed, -np.ones(n), np.ones(n)], (np.r_[t,t,t,t], np.r_[n+t,2*n+t,3*n+t,3*n+t+1])), shape=(n, m)).tocsr()
    rows, slots = np.arange(s*n), np.tile(t, s)
    scenario_ub = coo_matrix((np.r_[-np.ones(s*n), np.ones(s*n), -np.ones(s*n), -np.ones(s*n)], (np.tile(rows, 4), np.r_[slots,n+slots,2*n+slots,h0+rows])), shape=(s*n, m)).tocsr()
    rhs = -a.ravel()

    if fixed_grid is None:
        eq, eq_rhs = storage_eq, np.zeros(n)
    else:
        base = np.asarray(fixed_grid, float)
        grid_eq = coo_matrix((np.r_[np.ones(n), -np.ones(n), np.ones(n)], (np.r_[t,t,t], np.r_[t,u0+t,v0+t])), shape=(n, m)).tocsr()
        eq, eq_rhs = vstack([storage_eq, grid_eq]).tocsr(), np.r_[np.zeros(n), base]
        obj[u0:v0] = CFG['adjustment_up_multiplier'] * p
        obj[v0:] = CFG['adjustment_down_multiplier'] * p
        if lock_from is not None:
            lower[lock_from:n], upper[lock_from:n] = base[lock_from:], base[lock_from:]
    return dict(n=n, s=s, m=m, h0=h0, u0=u0, v0=v0, obj=obj, lower=lower, upper=upper, eq=eq, eq_rhs=eq_rhs, ub=scenario_ub, ub_rhs=rhs, mc=mc, md=md, p=p)


def add_cvar_risk(parts, weights, risk_lambda):
    """在原规划后添加 z、xi：CVaR = z + sum(w*xi)/(1-alpha)。"""
    if risk_lambda <= 0:
        return parts
    s, n, old_m = parts['s'], parts['n'], parts['m']
    weights = np.asarray(weights, float)
    require(len(weights) == s and np.isclose(weights.sum(), 1), '情景概率不合法')
    z, xi0, new_m = old_m, old_m + 1, old_m + 1 + s
    pad = csr_matrix((parts['eq'].shape[0], new_m - old_m))
    parts['eq'] = hstack([parts['eq'], pad]).tocsr()
    parts['obj'] = np.r_[parts['obj'], risk_lambda, risk_lambda * weights / (1 - CFG['cvar_alpha'])]
    parts['lower'] = np.r_[parts['lower'], -np.inf, np.zeros(s)]
    parts['upper'] = np.r_[parts['upper'], np.inf, np.full(s, np.inf)]
    # sum_t emergency_price[t]*h[s,t] - z - xi[s] <= 0
    rr, cc, dd = [], [], []
    loss_coef = CFG['emergency_multiplier'] * parts['p']
    for j in range(s):
        rr.extend([j] * n + [j, j])
        cc.extend((parts['h0'] + j*n + np.arange(n)).tolist() + [z, xi0+j])
        dd.extend(loss_coef.tolist() + [-1.0, -1.0])
    cvar_ub = coo_matrix((dd, (rr, cc)), shape=(s, new_m)).tocsr()
    parts['ub'] = vstack([hstack([parts['ub'], csr_matrix((parts['ub'].shape[0], new_m-old_m))]), cvar_ub]).tocsr()
    parts['ub_rhs'] = np.r_[parts['ub_rhs'], np.zeros(s)]
    parts['m'] = new_m
    parts['risk'] = {'z': z, 'xi0': xi0, 'weights': weights}
    return parts


def solve(parts):
    bounds = np.column_stack([parts['lower'], parts['upper']])
    result = linprog(parts['obj'], A_ub=parts['ub'], b_ub=parts['ub_rhs'], A_eq=parts['eq'], b_eq=parts['eq_rhs'], bounds=bounds, method='highs')
    require(result.success, 'LP求解失败：' + result.message)
    x, solver, n, m = result.x, 'LP', parts['n'], parts['m']
    simultaneous = (x[n:2*n] > CFG['tolerance']) & (x[2*n:3*n] > CFG['tolerance'])
    if simultaneous.any():
        t = np.arange(n)
        mode = coo_matrix((np.r_[np.ones(n), -parts['mc']*np.ones(n), np.ones(n), parts['md']*np.ones(n)], (np.r_[t,t,n+t,n+t], np.r_[n+t,m+t,2*n+t,m+t])), shape=(2*n, m+n)).tocsr()
        result = milp(np.r_[parts['obj'], np.zeros(n)], integrality=np.r_[np.zeros(m), np.ones(n)], bounds=Bounds(np.r_[parts['lower'], np.zeros(n)], np.r_[parts['upper'], np.ones(n)]), constraints=[LinearConstraint(hstack([parts['eq'], csr_matrix((parts['eq'].shape[0], n))]), parts['eq_rhs'], parts['eq_rhs']), LinearConstraint(hstack([parts['ub'], csr_matrix((parts['ub'].shape[0], n))]), -np.inf, parts['ub_rhs']), LinearConstraint(mode, -np.inf, np.r_[np.zeros(n), np.full(n, parts['md'])])], options={'mip_rel_gap': 1e-8})
        require(result.success, 'MILP求解失败：' + result.message)
        x, solver = result.x[:m], 'MILP'
    require(not (((x[n:2*n] > 1e-6) & (x[2*n:3*n] > 1e-6)).any()), '出现同时充放电')
    return x, solver


def make_plan(scenarios, weights, p, state, base_grid=None, risk_lambda=0.0):
    parts = _storage_parts(scenarios, p, state, fixed_grid=base_grid, lock_from=min(36, len(p)) if base_grid is not None else None)
    parts = add_cvar_risk(parts, weights, risk_lambda)
    x, solver = solve(parts)
    return np.maximum(x[:parts['n']], 0), solver


def execute_actual(grid, load_kwh, pv_kwh, state, battery=True):
    e = np.empty(len(grid)+1); e[0] = state
    charge = np.zeros(len(grid)); discharge = np.zeros(len(grid)); emergency = np.zeros(len(grid)); unused = np.zeros(len(grid))
    mc = CFG['max_charge_power_kw']/6 if battery else 0; md = CFG['max_discharge_power_kw']/6 if battery else 0
    for t in range(len(grid)):
        residual = grid[t] + pv_kwh[t] - load_kwh[t]
        if residual >= 0:
            charge[t] = min(residual, mc, max(0, (CFG['storage_max_kwh']-e[t])/CFG['charge_efficiency'])); unused[t] = residual-charge[t]
        else:
            discharge[t] = min(-residual, md, max(0, (e[t]-CFG['storage_min_kwh'])*CFG['discharge_efficiency'])); emergency[t] = -residual-discharge[t]
        e[t+1] = e[t] + CFG['charge_efficiency']*charge[t] - discharge[t]/CFG['discharge_efficiency']
    return dict(charge=charge, discharge=discharge, emergency=emergency, unused=unused, storage=e)


def adjustment_fee(updated, base, p):
    return p * (CFG['adjustment_up_multiplier']*np.maximum(updated-base, 0) + CFG['adjustment_down_multiplier']*np.maximum(base-updated, 0))


In [5]:
# 经济性回归：低概率高需求不应诱发免费超购；日内不应重复计原计划费。
def verify_purchase_cost_objective():
    original_cfg = CFG.copy()
    try:
        CFG['max_charge_power_kw'] = CFG['max_discharge_power_kw'] = 0.0
        CFG['planned_terminal_kwh'] = 6000.0
        scenarios = np.array([[100.0]] * 10 + [[1000.0]])
        p = np.ones(1)
        grid, _ = make_plan(scenarios, np.ones(11)/11, p, 6000.0)
        updated, _ = make_plan(np.array([[80.0]]), np.ones(1), p, 6000.0, np.array([100.0]))
        # 购100度：期望费用100+5*900/11；购1000度：费用1000。
        assert np.allclose(grid, [100.0]), '零点规划遗漏普通购电费'
        expected_bill = float(p @ grid + 5 * np.maximum(scenarios-grid, 0).mean())
        assert np.isclose(expected_bill, 100 + 5*900/11)
        # 原计划100度、需求80度时，减购还要付费，保留计划才是最优。
        assert np.allclose(updated, [100.0]), '日内规划重复计入原计划购电费'
    finally:
        CFG.update(original_cfg)
    print('普通购电成本与日内结算回归检查通过。')

verify_purchase_cost_objective()


普通购电成本与日内结算回归检查通过。


## 4. 三种策略的全年因果回测

所有策略在零点使用相同 Q2 初始计划。概率策略只在 06:00、12:00 对后续 6 小时启用“期望费用 + CVaR”优化；真实负荷与光伏只在每个 10 分钟执行结算时读取。

In [6]:
def run_strategy(name, kind, progress_every=60):
    state, daily_rows, detail_frames, decisions = CFG['initial_storage_kwh'], [], [], []
    for k, day in enumerate(dates):
        day_start = state
        scenarios00, weights00 = q2_initial_scenarios(k)
        base_grid, _ = make_plan(scenarios00, weights00, prices, state)
        updated = base_grid.copy(); blocks = []
        for start in (0, 36, 72, 108):
            end, hour = start+36, start//6
            if kind != 'none' and hour in (6, 12):
                scenarios, weights = update_scenarios(k, hour, perfect=(kind == 'perfect'))
                risk = CFG['risk_lambda'] if kind == 'probabilistic' else 0.0
                candidate, solver = make_plan(scenarios, weights, prices[start:], state, base_grid[start:], risk)
                updated[start:end] = candidate[:36]
                decisions.append({'策略': name, 'date': day.date(), '更新时间': f'{hour:02d}:00', '情景数': len(weights), '最大情景概率': float(weights.max()), '风险系数λ': risk, '调整电量(kWh)': float(np.abs(candidate[:36]-base_grid[start:end]).sum()), '求解器': solver})
            execution = execute_actual(updated[start:end], load_kw[k,start:end]/6, pv_kw[k,start:end]/6, state, battery=(k > 0))
            state = float(execution['storage'][-1]); blocks.append(execution)
        charge = np.concatenate([b['charge'] for b in blocks]); discharge = np.concatenate([b['discharge'] for b in blocks])
        emergency = np.concatenate([b['emergency'] for b in blocks]); unused = np.concatenate([b['unused'] for b in blocks])
        storage_start = np.concatenate([b['storage'][:-1] for b in blocks]); storage_end = np.concatenate([b['storage'][1:] for b in blocks])
        adj = adjustment_fee(updated, base_grid, prices)
        plan_fee, adj_fee, emerg_fee = float(prices@base_grid), float(adj.sum()), float(CFG['emergency_multiplier']*(prices@emergency))
        daily_rows.append({'策略': name, 'date': day.date(), '计划购电量(kWh)': base_grid.sum(), '最终计划购电量(kWh)': updated.sum(), '增购量(kWh)': np.maximum(updated-base_grid,0).sum(), '减购量(kWh)': np.maximum(base_grid-updated,0).sum(), '紧急购电量(kWh)': emergency.sum(), '未利用电量(kWh)': unused.sum(), '计划费用(元)': plan_fee, '调整费用(元)': adj_fee, '紧急费用(元)': emerg_fee, '总费用(元)': plan_fee+adj_fee+emerg_fee, '期初储能(kWh)': day_start, '期末储能(kWh)': state, '发生紧急购电': emergency.sum() > 1e-6})
        times = pd.date_range(day, periods=144, freq='10min')
        detail_frames.append(pd.DataFrame({'策略':name, 'date':day.date(), 'slot':np.arange(1,145), 'interval_start':times, 'price_yuan_per_kwh':prices, 'actual_load_kwh':load_kw[k]/6, 'actual_pv_kwh':pv_kw[k]/6, 'planned_grid_kwh':base_grid, 'updated_grid_kwh':updated, 'charge_kwh':charge, 'discharge_kwh':discharge, 'emergency_kwh':emergency, 'unused_kwh':unused, 'storage_start_kwh':storage_start, 'storage_end_kwh':storage_end, 'planned_cost_yuan':prices*base_grid, 'adjustment_cost_yuan':adj, 'emergency_cost_yuan':CFG['emergency_multiplier']*prices*emergency, 'total_cost_yuan':prices*base_grid+adj+CFG['emergency_multiplier']*prices*emergency, 'evaluation':day.date() >= pd.Timestamp('2025-02-01').date()}))
        if (k+1) % progress_every == 0 or k == len(dates)-1: print(f'{name}: {k+1}/{len(dates)} 天完成')
    return pd.DataFrame(daily_rows), pd.concat(detail_frames, ignore_index=True), pd.DataFrame(decisions)


started = time.perf_counter()
results = {name: run_strategy(name, kind) for name, kind in STRATEGIES.items()}
print(f'回测完成，用时 {time.perf_counter()-started:.1f} 秒。')


对照：00时Q2计划不更新: 60/365 天完成


对照：00时Q2计划不更新: 120/365 天完成


对照：00时Q2计划不更新: 180/365 天完成


对照：00时Q2计划不更新: 240/365 天完成


对照：00时Q2计划不更新: 300/365 天完成


...

方案：06时+12时概率风险更新: 180/365 天完成


方案：06时+12时概率风险更新: 240/365 天完成


方案：06时+12时概率风险更新: 300/365 天完成


方案：06时+12时概率风险更新: 360/365 天完成


方案：06时+12时概率风险更新: 365/365 天完成
回测完成，用时 49.1 秒。


## 5. 约束校验、输出文件与费用比较

In [7]:
def verify(detail):
    x = detail[detail['evaluation']]
    balance = x['updated_grid_kwh']+x['actual_pv_kwh']+x['discharge_kwh']+x['emergency_kwh']-x['actual_load_kwh']-x['charge_kwh']-x['unused_kwh']
    storage = x['storage_end_kwh']-x['storage_start_kwh']-CFG['charge_efficiency']*x['charge_kwh']+x['discharge_kwh']/CFG['discharge_efficiency']
    ans = {'供需平衡最大残差':abs(balance).max(), '储能更新最大残差':abs(storage).max(), '储能下界违约':max(0, CFG['storage_min_kwh']-x['storage_end_kwh'].min()), '储能上界违约':max(0, x['storage_end_kwh'].max()-CFG['storage_max_kwh']), '同时充放电时段数':int(((x['charge_kwh']>1e-6)&(x['discharge_kwh']>1e-6)).sum())}
    assert ans['供需平衡最大残差'] < 1e-5 and ans['储能更新最大残差'] < 1e-5
    assert ans['储能下界违约'] < 1e-5 and ans['储能上界违约'] < 1e-5 and ans['同时充放电时段数'] == 0
    return ans


rows, checks = [], {}
for name, (daily, detail, decisions) in results.items():
    x = daily[daily['date'] >= pd.Timestamp('2025-02-01').date()]
    row = x[['计划购电量(kWh)','最终计划购电量(kWh)','增购量(kWh)','减购量(kWh)','紧急购电量(kWh)','未利用电量(kWh)','计划费用(元)','调整费用(元)','紧急费用(元)','总费用(元)']].sum().to_dict()
    row.update({'策略':name, '发生紧急购电天数':int(x['发生紧急购电'].sum()), '更新次数':0 if decisions.empty else len(decisions[decisions['date'] >= pd.Timestamp('2025-02-01').date()]), '期初储能(kWh)':x['期初储能(kWh)'].iloc[0], '期末储能(kWh)':x['期末储能(kWh)'].iloc[-1]})
    rows.append(row); checks[name] = verify(detail)
    daily.to_csv(OUTPUT_DIR/f'{name}_daily_results.csv', index=False, encoding='utf-8-sig')
    detail.to_csv(OUTPUT_DIR/f'{name}_detail_10min.csv', index=False, encoding='utf-8-sig')
    decisions.to_csv(OUTPUT_DIR/f'{name}_decisions.csv', index=False, encoding='utf-8-sig')

summary = pd.DataFrame(rows)
base = summary.loc[summary['策略'].eq('对照：00时Q2计划不更新'), '总费用(元)'].iloc[0]
summary['相对不更新节省(元)'] = base-summary['总费用(元)']
summary['相对不更新节省比例'] = summary['相对不更新节省(元)']/base
summary.to_csv(OUTPUT_DIR/'summary.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(checks).T.to_csv(OUTPUT_DIR/'checks.csv', encoding='utf-8-sig')
display(pd.DataFrame(checks).T)
print('已保存：', OUTPUT_DIR)


,供需平衡最大残差,储能更新最大残差,储能下界违约,储能上界违约,同时充放电时段数
对照：00时Q2计划不更新,2.273737e-13,9.094947e-13,0.0,0.0,0.0
理论上限：06时+12时完美更新,2.273737e-13,9.094947e-13,0.0,0.0,0.0
方案：06时+12时概率风险更新,2.273737e-13,9.094947e-13,0.0,0.0,0.0


已保存： /Users/jinyu/workspace/MCM/CUMCM_2026/.tmp/q3_02_outputs


## 6. 评价期总费用

最后一列的概率方案是可实施回测结果；完美更新只作为预测准确时的理论上限。

In [8]:
from pathlib import Path
import pandas as pd

try:
    _summary = summary.copy()
except NameError:
    _root = Path.cwd()
    while not (_root / 'C题.md').exists() and _root != _root.parent:
        _root = _root.parent
    _summary = pd.read_csv(_root / '.tmp' / 'q3_02_outputs' / 'summary.csv')

total_fee_result = _summary[['策略','总费用(元)','计划费用(元)','调整费用(元)','紧急费用(元)','紧急购电量(kWh)','发生紧急购电天数','更新次数','相对不更新节省(元)','相对不更新节省比例']].copy()
display(total_fee_result.style.format({'总费用(元)':'{:,.2f}','计划费用(元)':'{:,.2f}','调整费用(元)':'{:,.2f}','紧急费用(元)':'{:,.2f}','紧急购电量(kWh)':'{:,.2f}','相对不更新节省(元)':'{:,.2f}','相对不更新节省比例':'{:.4%}'}))
total_fee_result


,策略,总费用(元),计划费用(元),调整费用(元),紧急费用(元),紧急购电量(kWh),发生紧急购电天数,更新次数,相对不更新节省(元),相对不更新节省比例
0,对照：00时Q2计划不更新,"13,631,409.27","13,285,607.79",0.00,"345,801.47","55,283.75",124,0,0.00,0.0000%
1,理论上限：06时+12时完美更新,"13,410,596.28","13,281,796.76","70,051.64","58,747.87","9,354.83",80,668,"220,812.99",1.6199%
2,方案：06时+12时概率风险更新,"13,705,565.47","13,279,871.65","337,281.20","88,412.62","14,481.48",93,668,"-74,156.20",-0.5440%


,策略,总费用(元),计划费用(元),调整费用(元),紧急费用(元),紧急购电量(kWh),发生紧急购电天数,更新次数,相对不更新节省(元),相对不更新节省比例
0,对照：00时Q2计划不更新,1.363141e+07,1.328561e+07,0.000000,345801.473580,55283.753848,124,0,0.000000,0.000000
1,理论上限：06时+12时完美更新,1.341060e+07,1.328180e+07,70051.644421,58747.870205,9354.832032,80,668,220812.992442,0.016199
2,方案：06时+12时概率风险更新,1.370557e+07,1.327987e+07,337281.202886,88412.616422,14481.475246,93,668,-74156.200377,-0.005440
